# News Topic Classification on AG News
## Sequential (BiLSTM) vs. Transformer Encoder — both implemented from scratch

**Case study — Transformer Modeling and Sequence Learning (Topic 2)**

**Research question.** *How do recurrent networks and Transformers compare at classifying short news
texts?*

**Dataset.** AG News (Kaggle: `amananandrai/ag-news-classification-dataset`) — 120,000 training and
7,600 test articles over four topics: World, Sports, Business, Sci/Tech. Following the brief, a
**controlled 60,000-record** stratified subset is used for training/validation, while the **full
official 7,600-record test set** is held back for the final comparison.

**What "from scratch" means here.** PyTorch is used for tensors, autograd and the optimiser.
`nn.Linear`, `nn.Embedding` and `nn.Dropout` are used purely as parameter containers. Every
sequence-modelling component is written out by hand:

| Component | Written by hand in this notebook |
|---|---|
| LSTM cell (input/forget/cell/output gates) | `LSTMCellScratch` |
| Recurrence over time + bidirectionality | `LSTMLayerScratch` |
| Scaled dot-product attention | `MultiHeadSelfAttentionScratch` |
| Multi-head split / merge projections | `MultiHeadSelfAttentionScratch` |
| Sinusoidal positional encoding | `sinusoidal_positional_encoding` |
| Layer normalisation | `LayerNormScratch` |
| Position-wise feed-forward + residual block | `EncoderBlockScratch` |

**Not used anywhere in the models:** `nn.LSTM`, `nn.GRU`, `nn.RNN`, `nn.MultiheadAttention`,
`nn.Transformer*`, `nn.LayerNorm`, `F.scaled_dot_product_attention`. `nn.LSTM` and
`F.layer_norm` appear **once each, inside Section 5**, used as a numerical reference to prove the
hand-written layers are correct.

**Structure.** 1 Data → 2 Controlled experiment setup → 3 BiLSTM from scratch →
4 Transformer from scratch → 5 Correctness tests → 6 Shared training harness →
7 Comparative evaluation → 8 Conclusion.

## 0. Configuration and reproducibility

`SMOKE_TEST = True` shrinks the whole notebook to a ~1 minute end-to-end run (2,000 records,
1 epoch) — useful for checking the notebook executes before committing to the full run.
Set it to `False` for the real experiment.

In [1]:
SMOKE_TEST = False   # True -> 2,000 records / 1 epoch, for a fast end-to-end check

# ---- experiment configuration -------------------------------------------------
SEED          = 42
SUBSET_SIZE   = 60_000     # controlled subset drawn from the 120k training file
VAL_FRACTION  = 0.10       # -> 54,000 train / 6,000 validation
MAX_LEN       = 64         # tokens per document (truncate / pad)
MIN_FREQ      = 2          # vocabulary cutoff, computed on the training split only

# ---- shared training budget (identical for both models) -----------------------
BATCH_SIZE    = 128
MAX_EPOCHS    = 8
PATIENCE      = 2          # early stopping on validation macro-F1
GRAD_CLIP     = 1.0
WEIGHT_DECAY  = 0.01

# ---- per-architecture learning rates (tuned on validation only) ---------------
LR_BILSTM      = 1e-3
LR_TRANSFORMER = 3e-4

# ---- architecture sizes -------------------------------------------------------
EMB_DIM        = 128       # shared embedding width, so both models see the same input capacity
LSTM_HIDDEN    = 128
LSTM_DROPOUT   = 0.30
D_MODEL        = 128
N_HEADS        = 4
N_LAYERS       = 2
D_FF           = 256
TRF_DROPOUT    = 0.10

if SMOKE_TEST:
    SUBSET_SIZE, MAX_EPOCHS, PATIENCE = 2_000, 1, 1

LABELS = ["World", "Sports", "Business", "Sci/Tech"]
N_CLASSES = len(LABELS)

DATA_DIR = "data/agnews"
ART_DIR  = "artifacts"

print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}  |  subset={SUBSET_SIZE:,}  epochs={MAX_EPOCHS}")

Mode: FULL RUN  |  subset=60,000  epochs=8


In [2]:
import json, math, os, random, re, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
Path(ART_DIR).mkdir(exist_ok=True)


def set_seed(seed: int = SEED) -> None:
    # Seed every source of randomness we touch, so the whole notebook is reproducible.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE} | threads: {torch.get_num_threads()}")

torch 2.15.0+cpu | device: cpu | threads: 6


In [3]:
# ---- plotting style ------------------------------------------------------------
# Categorical palette in fixed slot order (blue, orange, aqua, yellow). Colours are
# assigned by entity (model / class) and never re-assigned when a chart's series change.
SERIES   = {"BiLSTM": "#2a78d6", "Transformer": "#eb6834"}
CLASS_C  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK      = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#8a8880"}
GRID     = "#e3e2de"
SURFACE  = "#fcfcfb"
# Single-hue sequential ramp (light -> dark blue) for magnitude, e.g. confusion matrices.
BLUE_SEQ = LinearSegmentedColormap.from_list(
    "blue_seq", ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.labelcolor": INK["secondary"],
    "axes.titlecolor": INK["primary"], "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 12, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK["secondary"], "ytick.color": INK["secondary"],
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.bottom": True, "ytick.left": True, "xtick.direction": "out",
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.alpha": 1.0,
    "legend.frameon": False, "legend.fontsize": 9, "legend.labelcolor": INK["secondary"],
    "lines.linewidth": 2.0, "lines.markersize": 8,
    "font.size": 10, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
})


def recede(ax, axis="y"):
    # Grid and axes stay recessive: one faint set of gridlines behind the data, nothing else.
    ax.grid(axis=axis, zorder=0)
    ax.set_axisbelow(True)
    return ax


def save_fig(fig, name):
    path = Path(ART_DIR) / name
    fig.savefig(path)
    print(f"saved -> {path}")

## 1. Data acquisition

The dataset is pulled through the Kaggle API so the source is reproducible and citable. The
newer Kaggle token format (`KGAT_...`) is read from `~/.kaggle/access_token` or the
`KAGGLE_API_TOKEN` environment variable.

The CSVs are downloaded into their own directory and only ever read with `pandas.read_csv` —
nothing from the download directory is executed.

In [4]:
def ensure_agnews(data_dir: str = DATA_DIR) -> tuple[Path, Path]:
    # Download AG News from Kaggle unless it is already on disk.
    train_csv, test_csv = Path(data_dir) / "train.csv", Path(data_dir) / "test.csv"
    if train_csv.exists() and test_csv.exists():
        print(f"Found cached dataset in {data_dir}/ - skipping download.")
        return train_csv, test_csv

    token_file = Path.home() / ".kaggle" / "access_token"
    legacy     = Path.home() / ".kaggle" / "kaggle.json"
    if not (token_file.exists() or legacy.exists() or os.environ.get("KAGGLE_API_TOKEN")):
        raise RuntimeError(
            "No Kaggle credentials found. Do one of:\n"
            "  mkdir -p ~/.kaggle && echo KGAT_xxx > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token\n"
            "  export KAGGLE_API_TOKEN=KGAT_xxx\n"
            "  (legacy) place username/key JSON at ~/.kaggle/kaggle.json with chmod 600\n"
            "Get a token from https://www.kaggle.com/settings")

    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    Path(data_dir).mkdir(parents=True, exist_ok=True)
    api.dataset_download_files("amananandrai/ag-news-classification-dataset",
                               path=data_dir, unzip=True, quiet=True)
    print(f"Downloaded AG News to {data_dir}/")
    return train_csv, test_csv


TRAIN_CSV, TEST_CSV = ensure_agnews()

Found cached dataset in data/agnews/ - skipping download.


In [5]:
# The raw CSVs carry literal escape artefacts from the original scrape, e.g.
# "dwindling\band of ultra-cynics" and "\n" inside Description. Left in place, the
# tokeniser would weld two real words into one junk type, so they are stripped first.
_ESCAPES = re.compile(r"\\[a-z]")


def load_split(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"Class Index": "class_index"})
    title = df["Title"].fillna("").astype(str)
    desc  = df["Description"].fillna("").astype(str)
    text  = (title.str.strip() + ". " + desc.str.strip())
    text  = text.str.replace(_ESCAPES, " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    out = pd.DataFrame({"text": text, "label": df["class_index"].astype(int) - 1})
    before = len(out)
    out = out[(out["text"].str.len() > 0) & out["label"].between(0, N_CLASSES - 1)].reset_index(drop=True)
    print(f"{csv_path.name}: {before:,} rows -> {len(out):,} usable ({before - len(out)} dropped)")
    return out


train_full = load_split(TRAIN_CSV)
test_df    = load_split(TEST_CSV)

print("\nExample records:")
for i in [0, 1, 2]:
    print(f"  [{LABELS[train_full.label[i]]:<9}] {train_full.text[i][:95]}...")

train.csv: 120,000 rows -> 120,000 usable (0 dropped)
test.csv: 7,600 rows -> 7,600 usable (0 dropped)

Example records:
  [Business ] Wall St. Bears Claw Back Into the Black (Reuters). Reuters - Short-sellers, Wall Street's dwind...
  [Business ] Carlyle Looks Toward Commercial Aerospace (Reuters). Reuters - Private investment firm Carlyle ...
  [Business ] Oil and Economy Cloud Stocks' Outlook (Reuters). Reuters - Soaring crude prices plus worries bo...


## 2. The controlled experiment

Three things make this a *controlled* comparison rather than two unrelated training runs:

1. **One stratified 60,000-record subset** (15,000 per class) drawn from the training file with a
   fixed seed, split 54,000 / 6,000 into train and validation.
2. **One shared preprocessing pipeline** — the same tokeniser, the same vocabulary (built on the
   training split only, so no leakage), the same `MAX_LEN`, and the same integer tensors are fed to
   both models.
3. **One shared training harness** (Section 6) with an identical budget: same batch size, same
   maximum epochs, same optimiser family, same early-stopping rule, same seed.

The only thing that differs between the two runs is **the architecture** (and its learning rate,
discussed in Section 6).

In [6]:
set_seed()

# Stratified subset: exactly SUBSET_SIZE // 4 records per class.
per_class = SUBSET_SIZE // N_CLASSES
class_samples = [train_full[train_full.label == c].sample(
                     n=min(per_class, int((train_full.label == c).sum())), random_state=SEED)
                 for c in range(N_CLASSES)]
subset = (pd.concat(class_samples)
            .sample(frac=1.0, random_state=SEED)      # shuffle so splits are not class-ordered
            .reset_index(drop=True))

train_df, val_df = train_test_split(
    subset, test_size=VAL_FRACTION, random_state=SEED, stratify=subset["label"])
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

dist = pd.DataFrame({
    "Train": train_df.label.value_counts().sort_index(),
    "Validation": val_df.label.value_counts().sort_index(),
    "Test (official)": test_df.label.value_counts().sort_index(),
}).rename(index=dict(enumerate(LABELS)))
dist.loc["TOTAL"] = dist.sum()
print("Class distribution across splits\n")
print(dist.to_string())
print(f"\nControlled subset: {len(subset):,} records -> {len(train_df):,} train / {len(val_df):,} val")
print(f"Held-out official test set: {len(test_df):,} records")

Class distribution across splits

          Train  Validation  Test (official)
label                                       
World     13500        1500             1900
Sports    13500        1500             1900
Business  13500        1500             1900
Sci/Tech  13500        1500             1900
TOTAL     54000        6000             7600

Controlled subset: 60,000 records -> 54,000 train / 6,000 val
Held-out official test set: 7,600 records


In [7]:
TOKEN_RE = re.compile(r"[a-z0-9']+")
PAD_ID, UNK_ID = 0, 1


def tokenize(text: str) -> list[str]:
    # Lowercase + keep alphanumeric/apostrophe runs. Deliberately simple: both models get the
    # same tokens, so tokeniser sophistication cannot favour one architecture over the other.
    return TOKEN_RE.findall(text.lower())


# Vocabulary from the TRAINING split only.
counts = Counter()
train_tokens = [tokenize(t) for t in train_df.text]
for toks in train_tokens:
    counts.update(toks)

itos = ["<pad>", "<unk>"] + [w for w, c in counts.most_common() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_SIZE = len(itos)

print(f"Token types seen in training split : {len(counts):,}")
print(f"Vocabulary (min_freq >= {MIN_FREQ})       : {VOCAB_SIZE:,}")
print(f"Most common: {[w for w, _ in counts.most_common(12)]}")

Token types seen in training split : 53,475
Vocabulary (min_freq >= 2)       : 33,964
Most common: ['the', 'to', 'a', 'of', 'in', 'and', 'on', 'for', '39', 's', 'that', 'with']


In [8]:
lengths = np.array([len(t) for t in train_tokens])
coverage = (lengths <= MAX_LEN).mean()

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.hist(lengths, bins=np.arange(0, 121, 2), color=SERIES["BiLSTM"], edgecolor=SURFACE, linewidth=0.6, zorder=2)
ax.axvline(MAX_LEN, color=INK["primary"], linestyle="--", linewidth=1.5, zorder=3)
ax.annotate(f"MAX_LEN = {MAX_LEN}\n{coverage:.1%} of documents fit untruncated",
            xy=(MAX_LEN, ax.get_ylim()[1] * 0.92), xytext=(MAX_LEN + 5, ax.get_ylim()[1] * 0.92),
            color=INK["primary"], fontsize=9, va="top")
ax.set(title="Document length in tokens (training split)", xlabel="tokens", ylabel="documents", xlim=(0, 120))
recede(ax)
fig.tight_layout(); save_fig(fig, "fig0_token_lengths.png"); plt.show()

print(f"median={np.median(lengths):.0f}  mean={lengths.mean():.1f}  p95={np.percentile(lengths, 95):.0f}  max={lengths.max()}")
print(f"Coverage at MAX_LEN={MAX_LEN}: {coverage:.2%} of training documents are not truncated.")

saved -> artifacts/fig0_token_lengths.png
median=39  mean=39.4  p95=58  max=181
Coverage at MAX_LEN=64: 96.94% of training documents are not truncated.


In [9]:
def encode(texts, max_len: int = MAX_LEN):
    # -> (ids[N, max_len] int64, lengths[N] int64). Right-padded with PAD_ID.
    n = len(texts)
    ids  = np.zeros((n, max_len), dtype=np.int64)
    lens = np.zeros(n, dtype=np.int64)
    for i, text in enumerate(texts):
        toks = tokenize(text)[:max_len]
        if not toks:                       # keep every row non-empty so masks never sum to 0
            toks = ["<unk>"]
        ids[i, :len(toks)] = [stoi.get(t, UNK_ID) for t in toks]
        lens[i] = len(toks)
    return torch.from_numpy(ids), torch.from_numpy(lens)


def make_dataset(df):
    ids, lens = encode(df.text.tolist())
    return TensorDataset(ids, lens, torch.tensor(df.label.values, dtype=torch.long))


t0 = time.time()
train_ds, val_ds, test_ds = make_dataset(train_df), make_dataset(val_df), make_dataset(test_df)
print(f"Encoded {len(train_ds) + len(val_ds) + len(test_ds):,} documents in {time.time() - t0:.1f}s")

g = torch.Generator().manual_seed(SEED)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g, drop_last=False)
val_dl   = DataLoader(val_ds,   batch_size=512, shuffle=False)
test_dl  = DataLoader(test_ds,  batch_size=512, shuffle=False)

xb, lb, yb = next(iter(train_dl))
print(f"batch ids {tuple(xb.shape)} | lengths {tuple(lb.shape)} | labels {tuple(yb.shape)}")
print(f"first row decoded: {' '.join(itos[i] for i in xb[0][:lb[0]])[:110]}...")

Encoded 67,600 documents in 0.5s
batch ids (128, 64) | lengths (128,) | labels (128,)
first row decoded: 'miracle baby' nurses face probe the nursing and <unk> council is investigating the activities of seven women ...


## 3. Model A — Bidirectional LSTM, written from scratch

The LSTM is built up in three layers of abstraction, none of which uses `nn.LSTM`:

**`LSTMCellScratch`** — one timestep. The four gates are computed in a single fused matmul and
split with `chunk(4)`:

$$\begin{aligned}
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) &\quad\text{(input gate)}\\
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) &\quad\text{(forget gate)}\\
g_t &= \tanh(W_g x_t + U_g h_{t-1} + b_g) &\quad\text{(candidate cell)}\\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) &\quad\text{(output gate)}\\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t, \qquad h_t = o_t \odot \tanh(c_t)
\end{aligned}$$

The gate order `i, f, g, o` deliberately matches PyTorch's own convention so that Section 5 can
copy `nn.LSTM`'s weights straight across and compare outputs numerically.

**`LSTMLayerScratch`** — the explicit Python loop over timesteps, in either direction. Two details
matter and are the usual source of silent accuracy loss:

1. *Padding must not corrupt the state.* Each step gates the update by the mask,
   `h ← m·h_new + (1−m)·h_prev`. Without this, pad tokens keep driving the recurrence and the
   final state drifts away from the real end of the sentence.
2. *The backward pass must start at the true end of each sequence.* Because padding sits at the
   right and masked steps are exact no-ops, iterating `t = T−1 … 0` with the same masking is
   equivalent to starting at each sequence's own last real token — so the returned final state is
   the correct per-sequence endpoint for both directions.

**`BiLSTMClassifier`** — embedding → forward + backward layers → pooling → linear head. Pooling
concatenates the masked mean over all timesteps with the final hidden state of each direction
(`2H + H + H = 4H` features): mean-pooling alone discards the endpoint the recurrence worked to
build, while the endpoint alone is noisy on documents this short.

In [10]:
class LSTMCellScratch(nn.Module):
    # One LSTM timestep. W: input->gates (with bias), U: hidden->gates.
    def __init__(self, d_in: int, d_hidden: int):
        super().__init__()
        self.d_hidden = d_hidden
        self.W = nn.Linear(d_in, 4 * d_hidden, bias=True)
        self.U = nn.Linear(d_hidden, 4 * d_hidden, bias=False)
        self.reset_parameters()

    def reset_parameters(self):
        # Classic LSTM init: U(-1/sqrt(H), 1/sqrt(H)) on every parameter.
        k = 1.0 / math.sqrt(self.d_hidden)
        for p in self.parameters():
            nn.init.uniform_(p, -k, k)

    def forward(self, x_t, h_prev, c_prev):
        gates = self.W(x_t) + self.U(h_prev)             # (B, 4H) in one matmul
        i, f, g, o = gates.chunk(4, dim=-1)              # PyTorch gate order: i, f, g, o
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_t = f * c_prev + i * g
        h_t = o * torch.tanh(c_t)
        return h_t, c_t


class LSTMLayerScratch(nn.Module):
    # Unrolls LSTMCellScratch over time in one direction, mask-aware.
    def __init__(self, d_in: int, d_hidden: int, reverse: bool = False):
        super().__init__()
        self.cell, self.d_hidden, self.reverse = LSTMCellScratch(d_in, d_hidden), d_hidden, reverse

    def forward(self, x, mask):
        # x (B, T, D); mask (B, T) float with 1.0 on real tokens.
        B, T, _ = x.shape
        h = x.new_zeros(B, self.d_hidden)
        c = x.new_zeros(B, self.d_hidden)
        steps = range(T - 1, -1, -1) if self.reverse else range(T)
        outputs = []
        for t in steps:
            m = mask[:, t].unsqueeze(1)                  # (B, 1)
            h_new, c_new = self.cell(x[:, t], h, c)
            # Gate the state update: a padded step leaves (h, c) exactly unchanged.
            h = m * h_new + (1.0 - m) * h
            c = m * c_new + (1.0 - m) * c
            outputs.append(h)
        if self.reverse:
            outputs.reverse()                            # back into position order
        return torch.stack(outputs, dim=1), h            # (B, T, H), (B, H) final valid state


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = EMB_DIM, hidden: int = LSTM_HIDDEN,
                 n_classes: int = N_CLASSES, dropout: float = LSTM_DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.1)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].zero_()
        self.forward_layer  = LSTMLayerScratch(emb_dim, hidden, reverse=False)
        self.backward_layer = LSTMLayerScratch(emb_dim, hidden, reverse=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(4 * hidden, n_classes)

    def forward(self, ids, lengths=None):
        mask = (ids != PAD_ID).float()                           # (B, T)
        x = self.embedding(ids)
        out_f, h_f = self.forward_layer(x, mask)
        out_b, h_b = self.backward_layer(x, mask)
        seq = torch.cat([out_f, out_b], dim=-1)                  # (B, T, 2H)
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        mean_pooled = (seq * mask.unsqueeze(-1)).sum(dim=1) / denom
        features = torch.cat([mean_pooled, h_f, h_b], dim=-1)    # (B, 4H)
        return self.classifier(self.dropout(features))


print(BiLSTMClassifier(VOCAB_SIZE))

BiLSTMClassifier(
  (embedding): Embedding(33964, 128, padding_idx=0)
  (forward_layer): LSTMLayerScratch(
    (cell): LSTMCellScratch(
      (W): Linear(in_features=128, out_features=512, bias=True)
      (U): Linear(in_features=128, out_features=512, bias=False)
    )
  )
  (backward_layer): LSTMLayerScratch(
    (cell): LSTMCellScratch(
      (W): Linear(in_features=128, out_features=512, bias=True)
      (U): Linear(in_features=128, out_features=512, bias=False)
    )
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=512, out_features=4, bias=True)
)


## 4. Model B — Transformer encoder, written from scratch

A pre-norm Transformer encoder, assembled from four hand-written pieces:

**`sinusoidal_positional_encoding`** — the original Vaswani et al. formulation, since
self-attention is permutation-invariant and would otherwise see a bag of words:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad
  PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

**`LayerNormScratch`** — normalises over the feature dimension with learned $\gamma, \beta$,
using the biased variance so it matches `F.layer_norm` exactly (verified in Section 5).

**`MultiHeadSelfAttentionScratch`** — scaled dot-product attention, with the head split and merge
done by hand:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}} + M\right)V$$

The key-padding mask $M$ is applied as a large negative constant (`-1e9`) rather than `-inf`:
with `-inf`, a row that is entirely masked would softmax to `NaN` and poison the gradients.

**`EncoderBlockScratch`** — pre-norm residuals, `x + attn(norm(x))` then `x + ffn(norm(x))`. Pre-norm
is the deliberate choice here: it trains stably without a learning-rate warmup schedule, which
matters when the budget is a handful of CPU epochs.

Capacity is deliberately matched to the BiLSTM so the comparison is architectural rather than a
disguised parameter-count contest — Section 7 reports both totals.

In [11]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    # (max_len, d_model) fixed, non-learned position signal.
    position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)        # (T, 1)
    even_i   = torch.arange(0, d_model, 2, dtype=torch.float32)               # (d_model/2,)
    div_term = torch.exp(-math.log(10000.0) * even_i / d_model)
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)[:, : pe[:, 1::2].shape[1]]
    return pe


class LayerNormScratch(nn.Module):
    # Normalises over the last dimension. Biased variance, to match F.layer_norm.
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mu  = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mu) / torch.sqrt(var + self.eps) + self.beta


class MultiHeadSelfAttentionScratch(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads, self.d_k = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, t):
        B, T, _ = t.shape
        return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)   # (B, h, T, d_k)

    def forward(self, x, key_mask=None, return_attention: bool = False):
        B, T, _ = x.shape
        q, k, v = self._split_heads(self.W_q(x)), self._split_heads(self.W_k(x)), self._split_heads(self.W_v(x))

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)     # (B, h, T, T)
        if key_mask is not None:
            # Broadcast (B, T) -> (B, 1, 1, T): every query ignores padded keys.
            scores = scores.masked_fill(key_mask[:, None, None, :] == 0, -1e9)

        attention = torch.softmax(scores, dim=-1)
        context = self.dropout(attention) @ v                        # (B, h, T, d_k)
        context = context.transpose(1, 2).reshape(B, T, self.n_heads * self.d_k)
        out = self.W_o(context)
        return (out, attention) if return_attention else out


class EncoderBlockScratch(nn.Module):
    # Pre-norm block: x + attn(norm(x)), then x + ffn(norm(x)).
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.norm1 = LayerNormScratch(d_model)
        self.attention = MultiHeadSelfAttentionScratch(d_model, n_heads, dropout)
        self.norm2 = LayerNormScratch(d_model)
        self.ff_in  = nn.Linear(d_model, d_ff)
        self.ff_out = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_mask=None):
        x = x + self.dropout(self.attention(self.norm1(x), key_mask))
        h = self.norm2(x)
        h = self.ff_out(self.dropout(F.gelu(self.ff_in(h))))
        return x + self.dropout(h)


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = D_MODEL, n_heads: int = N_HEADS,
                 n_layers: int = N_LAYERS, d_ff: int = D_FF, max_len: int = MAX_LEN,
                 n_classes: int = N_CLASSES, dropout: float = TRF_DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.1)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].zero_()
        self.register_buffer("pos_encoding", sinusoidal_positional_encoding(max_len, d_model))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            EncoderBlockScratch(d_model, n_heads, d_ff, dropout) for _ in range(n_layers))
        self.norm_final = LayerNormScratch(d_model)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, ids, lengths=None, return_attention: bool = False):
        mask = (ids != PAD_ID)                                        # (B, T) bool
        T = ids.size(1)
        # Scale embeddings by sqrt(d_model) so they are not swamped by the positional signal.
        x = self.embedding(ids) * math.sqrt(self.d_model) + self.pos_encoding[:T].unsqueeze(0)
        x = self.dropout(x)

        attn_maps = []
        for block in self.blocks:
            if return_attention:
                normed = block.norm1(x)
                attended, attn = block.attention(normed, mask, return_attention=True)
                attn_maps.append(attn)
                x = x + block.dropout(attended)
                h = block.ff_out(block.dropout(F.gelu(block.ff_in(block.norm2(x)))))
                x = x + block.dropout(h)
            else:
                x = block(x, mask)
        x = self.norm_final(x)

        m = mask.float().unsqueeze(-1)
        pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)     # masked mean over tokens
        logits = self.classifier(self.dropout(pooled))
        return (logits, attn_maps) if return_attention else logits


def count_parameters(model) -> dict:
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    emb   = sum(p.numel() for n, p in model.named_parameters() if "embedding" in n)
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


print(TransformerClassifier(VOCAB_SIZE))

TransformerClassifier(
  (embedding): Embedding(33964, 128, padding_idx=0)
  (dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-1): 2 x EncoderBlockScratch(
      (norm1): LayerNormScratch()
      (attention): MultiHeadSelfAttentionScratch(
        (W_q): Linear(in_features=128, out_features=128, bias=True)
        (W_k): Linear(in_features=128, out_features=128, bias=True)
        (W_v): Linear(in_features=128, out_features=128, bias=True)
        (W_o): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): LayerNormScratch()
      (ff_in): Linear(in_features=128, out_features=256, bias=True)
      (ff_out): Linear(in_features=256, out_features=128, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm_final): LayerNormScratch()
  (classifier): Linear(in_features=128, out_features=4, bias=True)
)


## 5. Correctness tests for the hand-written layers

A from-scratch layer that is subtly wrong still trains and still produces a plausible-looking
accuracy number — it just quietly produces a *worse* one, which would invalidate the whole
comparison. These checks are therefore part of the deliverable, not scaffolding.

| Check | What it rules out |
|---|---|
| `LSTMCellScratch` vs. `nn.LSTM` with copied weights | wrong gate order, wrong gate algebra, misplaced bias |
| `LayerNormScratch` vs. `F.layer_norm` | unbiased-vs-biased variance, wrong reduction axis |
| `MultiHeadSelfAttentionScratch` vs. a naive per-head loop | wrong head split/merge reshape, wrong scaling |
| attention rows sum to 1 | softmax over the wrong axis |
| **padding invariance** | the two masking bugs called out in Section 3 |
| overfitting 200 examples | a broken backward path / dead gradients |

> `nn.LSTM` and `F.layer_norm` appear **only in this section**, purely as numerical references.
> Neither is used by either model.

In [12]:
set_seed(0)
results = {}

# ---- Check 1: LSTMCellScratch / LSTMLayerScratch vs. nn.LSTM ------------------
# nn.LSTM is used HERE ONLY as a reference implementation, never inside our models.
B, T, D, H = 4, 7, 10, 16
ref_lstm = nn.LSTM(D, H, num_layers=1, batch_first=True, bidirectional=False)
ours = LSTMLayerScratch(D, H, reverse=False)
with torch.no_grad():
    # nn.LSTM packs gates as [i, f, g, o] - the same order LSTMCellScratch chunks into.
    ours.cell.W.weight.copy_(ref_lstm.weight_ih_l0)
    ours.cell.W.bias.copy_(ref_lstm.bias_ih_l0 + ref_lstm.bias_hh_l0)   # our U has no bias
    ours.cell.U.weight.copy_(ref_lstm.weight_hh_l0)

x = torch.randn(B, T, D)
full_mask = torch.ones(B, T)
with torch.no_grad():
    ref_out, (ref_h, _) = ref_lstm(x)
    our_out, our_h = ours(x, full_mask)

d_out, d_h = (ref_out - our_out).abs().max().item(), (ref_h[0] - our_h).abs().max().item()
results["LSTM cell matches nn.LSTM"] = (max(d_out, d_h) < 1e-5, f"max|diff| outputs={d_out:.2e} final h={d_h:.2e}")

# ---- Check 2: LayerNormScratch vs. F.layer_norm --------------------------------
ln = LayerNormScratch(32)
with torch.no_grad():
    nn.init.normal_(ln.gamma, 1.0, 0.2)
    nn.init.normal_(ln.beta, 0.0, 0.2)
    z = torch.randn(5, 9, 32) * 3.0 + 1.5
    d_ln = (ln(z) - F.layer_norm(z, (32,), ln.gamma, ln.beta, eps=1e-5)).abs().max().item()
results["LayerNorm matches F.layer_norm"] = (d_ln < 1e-6, f"max|diff|={d_ln:.2e}")

In [13]:
# ---- Check 3: multi-head attention vs. a naive per-head loop -------------------
# The reference recomputes attention one (batch, head) pair at a time with explicit
# column slicing, which independently verifies the view/transpose head split and merge.
set_seed(0)
d_model, n_heads, Tq = 24, 3, 6
mha = MultiHeadSelfAttentionScratch(d_model, n_heads, dropout=0.0).eval()
x = torch.randn(3, Tq, d_model)
km = torch.ones(3, Tq, dtype=torch.bool)
km[0, 4:] = False          # first row has two padded positions
km[2, 5:] = False

with torch.no_grad():
    out, attn = mha(x, km, return_attention=True)
    q_all, k_all, v_all = mha.W_q(x), mha.W_k(x), mha.W_v(x)
    d_k = d_model // n_heads
    naive = torch.zeros_like(out)
    for b in range(x.size(0)):
        head_outputs = []
        for h in range(n_heads):
            sl = slice(h * d_k, (h + 1) * d_k)
            qh, kh, vh = q_all[b, :, sl], k_all[b, :, sl], v_all[b, :, sl]
            s = (qh @ kh.T) / math.sqrt(d_k)
            s = s.masked_fill(~km[b].unsqueeze(0), -1e9)
            head_outputs.append(torch.softmax(s, dim=-1) @ vh)
        naive[b] = mha.W_o(torch.cat(head_outputs, dim=-1))

d_attn = (out - naive).abs().max().item()
results["Attention matches naive per-head reference"] = (d_attn < 1e-5, f"max|diff|={d_attn:.2e}")

row_sums = attn.sum(dim=-1)
results["Attention rows sum to 1"] = (
    torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5),
    f"max|rowsum-1|={(row_sums - 1).abs().max().item():.2e}")

# Padded keys must receive essentially zero attention mass from every query.
pad_mass = attn[0, :, :, 4:].max().item()
results["Padded keys get ~zero attention"] = (pad_mass < 1e-6, f"max mass on a padded key={pad_mass:.2e}")

In [14]:
# ---- Check 4: padding invariance (the decisive masking test) -------------------
# Appending extra <pad> tokens must not move either model's logits by more than float noise.
set_seed(0)
bilstm_t = BiLSTMClassifier(VOCAB_SIZE).eval()
trf_t    = TransformerClassifier(VOCAB_SIZE).eval()

short_ids = torch.randint(2, 500, (6, 12))
short_ids[0, 8:] = PAD_ID          # mixed real lengths within the batch
short_ids[3, 5:] = PAD_ID
padded_ids = torch.cat([short_ids, torch.full((6, 20), PAD_ID, dtype=torch.long)], dim=1)

with torch.no_grad():
    for name, model in [("BiLSTM", bilstm_t), ("Transformer", trf_t)]:
        d = (model(short_ids) - model(padded_ids)).abs().max().item()
        results[f"{name}: logits unchanged by extra padding"] = (d < 1e-4, f"max|diff|={d:.2e}")

# ---- Check 5: both models can overfit 200 examples (backward path is alive) ----
sub_ids, sub_lens, sub_y = train_ds[:200]
for name, model, lr in [("BiLSTM", BiLSTMClassifier(VOCAB_SIZE), 3e-3),
                        ("Transformer", TransformerClassifier(VOCAB_SIZE), 3e-3)]:
    set_seed()
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(30):
        opt.zero_grad()
        loss = F.cross_entropy(model(sub_ids, sub_lens), sub_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(sub_ids, sub_lens).argmax(-1) == sub_y).float().mean().item()
    results[f"{name}: overfits 200 examples"] = (acc > 0.95, f"train accuracy={acc:.1%} after 30 steps")

# ---- report -------------------------------------------------------------------
print(f"{'check':<48} {'result':<7} detail")
print("-" * 100)
failed = 0
for name, (ok, detail) in results.items():
    failed += not ok
    print(f"{name:<48} {'PASS' if ok else 'FAIL':<7} {detail}")
print("-" * 100)
assert failed == 0, f"{failed} correctness check(s) failed - fix before training."
print(f"All {len(results)} checks passed. The hand-written layers are numerically correct.")

del bilstm_t, trf_t, ref_lstm, ours, mha

check                                            result  detail
----------------------------------------------------------------------------------------------------
LSTM cell matches nn.LSTM                        PASS    max|diff| outputs=5.96e-08 final h=4.47e-08
LayerNorm matches F.layer_norm                   PASS    max|diff|=4.77e-07
Attention matches naive per-head reference       PASS    max|diff|=8.94e-08
Attention rows sum to 1                          PASS    max|rowsum-1|=1.79e-07
Padded keys get ~zero attention                  PASS    max mass on a padded key=0.00e+00
BiLSTM: logits unchanged by extra padding        PASS    max|diff|=0.00e+00
Transformer: logits unchanged by extra padding   PASS    max|diff|=1.19e-07
BiLSTM: overfits 200 examples                    PASS    train accuracy=100.0% after 30 steps
Transformer: overfits 200 examples               PASS    train accuracy=100.0% after 30 steps
-----------------------------------------------------------------------

## 6. Shared training harness

Both models go through the *same* `train_model` function, so the training budget is controlled:
identical batch size, maximum epochs, optimiser family, weight decay, gradient clipping,
early-stopping rule and seed.

**On the one thing that is not identical — the learning rate.** The BiLSTM runs at `1e-3` and the
Transformer at `3e-4`, each picked on the validation split. Forcing a single shared learning rate
would *look* fairer but would actually handicap whichever architecture it suited less, and would
measure "sensitivity to this particular learning rate" rather than the architectures themselves.
The controlled quantity here is the compute budget; the learning rate is tuned per model, and never
on the test set.

In [15]:
@torch.no_grad()
def evaluate(model, dataloader) -> dict:
    model.eval()
    total_loss, preds, targets = 0.0, [], []
    for ids, lens, y in dataloader:
        ids, lens, y = ids.to(DEVICE), lens.to(DEVICE), y.to(DEVICE)
        logits = model(ids, lens)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        preds.append(logits.argmax(dim=-1).cpu())
        targets.append(y.cpu())
    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return {"loss": total_loss / len(targets),
            "accuracy": accuracy_score(targets, preds),
            "macro_f1": f1_score(targets, preds, average="macro"),
            "weighted_f1": f1_score(targets, preds, average="weighted"),
            "preds": preds, "targets": targets}


def train_model(model, name: str, lr: float, max_epochs: int = MAX_EPOCHS,
                patience: int = PATIENCE) -> dict:
    set_seed()                                   # same initial shuffling for both models
    model = model.to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [],
               "val_macro_f1": [], "epoch_seconds": []}
    best_f1, best_state, best_epoch, stale = -1.0, None, 0, 0
    n_batches = len(train_dl)

    params = count_parameters(model)
    print(f"=== {name} | lr={lr:g} | {params['total']:,} params "
          f"({params['non_embedding']:,} excluding embeddings) ===")

    wall_start = time.time()
    for epoch in range(1, max_epochs + 1):
        model.train()
        t0, run_loss, run_correct, seen = time.time(), 0.0, 0, 0
        for step, (ids, lens, y) in enumerate(train_dl, 1):
            ids, lens, y = ids.to(DEVICE), lens.to(DEVICE), y.to(DEVICE)
            optimiser.zero_grad(set_to_none=True)
            logits = model(ids, lens)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimiser.step()

            run_loss += loss.item() * y.size(0)
            run_correct += (logits.argmax(dim=-1) == y).sum().item()
            seen += y.size(0)
            if step % max(1, n_batches // 4) == 0 or step == n_batches:
                print(f"  epoch {epoch} [{step:>4}/{n_batches}] "
                      f"train loss {run_loss / seen:.4f} acc {run_correct / seen:.4f}", flush=True)

        elapsed = time.time() - t0
        val = evaluate(model, val_dl)
        history["train_loss"].append(run_loss / seen)
        history["train_acc"].append(run_correct / seen)
        history["val_loss"].append(val["loss"])
        history["val_acc"].append(val["accuracy"])
        history["val_macro_f1"].append(val["macro_f1"])
        history["epoch_seconds"].append(elapsed)
        print(f"  -> epoch {epoch}: val loss {val['loss']:.4f} | val acc {val['accuracy']:.4f} "
              f"| val macro-F1 {val['macro_f1']:.4f} | {elapsed:.1f}s")

        if val["macro_f1"] > best_f1:
            best_f1, best_epoch, stale = val["macro_f1"], epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= patience:
                print(f"  early stopping: no val macro-F1 gain for {patience} epoch(s)")
                break

    total_seconds = time.time() - wall_start
    model.load_state_dict(best_state)                       # restore the best checkpoint
    torch.save(best_state, Path(ART_DIR) / f"{name.lower()}_best.pt")
    print(f"  best epoch {best_epoch} (val macro-F1 {best_f1:.4f}) | total {total_seconds:.1f}s\n")

    return {"model": model, "name": name, "history": history, "params": params,
            "lr": lr, "best_epoch": best_epoch, "best_val_macro_f1": best_f1,
            "total_seconds": total_seconds,
            "seconds_per_epoch": float(np.mean(history["epoch_seconds"]))}

In [16]:
# ---- Train Model A: BiLSTM ----------------------------------------------------
set_seed()
run_bilstm = train_model(BiLSTMClassifier(VOCAB_SIZE), "BiLSTM", LR_BILSTM)

=== BiLSTM | lr=0.001 | 4,612,612 params (265,220 excluding embeddings) ===


  epoch 1 [ 105/422] train loss 0.9726 acc 0.5602


  epoch 1 [ 210/422] train loss 0.6760 acc 0.7186


  epoch 1 [ 315/422] train loss 0.5519 acc 0.7793


  epoch 1 [ 420/422] train loss 0.4876 acc 0.8114


...

  epoch 3 [ 210/422] train loss 0.1109 acc 0.9649


  epoch 3 [ 315/422] train loss 0.1164 acc 0.9629


  epoch 3 [ 420/422] train loss 0.1203 acc 0.9615


  epoch 3 [ 422/422] train loss 0.1203 acc 0.9615


  -> epoch 3: val loss 0.3132 | val acc 0.9012 | val macro-F1 0.9010 | 40.8s
  early stopping: no val macro-F1 gain for 2 epoch(s)
  best epoch 1 (val macro-F1 0.9091) | total 126.0s



In [17]:
# ---- Train Model B: Transformer encoder ---------------------------------------
set_seed()
run_transformer = train_model(TransformerClassifier(VOCAB_SIZE), "Transformer", LR_TRANSFORMER)

=== Transformer | lr=0.0003 | 4,613,124 params (265,732 excluding embeddings) ===


  epoch 1 [ 105/422] train loss 1.0881 acc 0.5435


  epoch 1 [ 210/422] train loss 0.7780 acc 0.6894


  epoch 1 [ 315/422] train loss 0.6427 acc 0.7500


  epoch 1 [ 420/422] train loss 0.5670 acc 0.7837


...

  epoch 5 [ 210/422] train loss 0.0884 acc 0.9699


  epoch 5 [ 315/422] train loss 0.0933 acc 0.9681


  epoch 5 [ 420/422] train loss 0.0957 acc 0.9669


  epoch 5 [ 422/422] train loss 0.0957 acc 0.9669


  -> epoch 5: val loss 0.3646 | val acc 0.9027 | val macro-F1 0.9026 | 68.1s
  early stopping: no val macro-F1 gain for 2 epoch(s)
  best epoch 3 (val macro-F1 0.9077) | total 353.3s



## 7. Comparative evaluation

Everything below is computed on the **full official 7,600-record test set**, which neither model
nor any hyperparameter choice has seen. Both models are evaluated from their restored
best-validation checkpoints.

In [18]:
RUNS = [run_bilstm, run_transformer]
test_eval = {}
for run in RUNS:
    test_eval[run["name"]] = evaluate(run["model"], test_dl)

headline = pd.DataFrame([{
    "Model": run["name"],
    "Test accuracy": test_eval[run["name"]]["accuracy"],
    "Macro F1": test_eval[run["name"]]["macro_f1"],
    "Weighted F1": test_eval[run["name"]]["weighted_f1"],
    "Test loss": test_eval[run["name"]]["loss"],
    "Best epoch": run["best_epoch"],
    "Val macro F1": run["best_val_macro_f1"],
} for run in RUNS]).set_index("Model")

print("Headline test-set results\n")
print(headline.round(4).to_string())

gap = (test_eval["Transformer"]["accuracy"] - test_eval["BiLSTM"]["accuracy"]) * 100
leader = "Transformer" if gap > 0 else "BiLSTM"
print(f"\nAccuracy gap: {abs(gap):.2f} percentage points in favour of the {leader}.")
print("Section 7.6 tests whether that gap is statistically distinguishable from noise.")

Headline test-set results

             Test accuracy  Macro F1  Weighted F1  Test loss  Best epoch  Val macro F1
Model                                                                                 
BiLSTM              0.9114    0.9111       0.9111     0.2797           1        0.9091
Transformer         0.9105    0.9104       0.9104     0.2944           3        0.9077

Accuracy gap: 0.09 percentage points in favour of the BiLSTM.
Section 7.6 tests whether that gap is statistically distinguishable from noise.


### 7.1 Per-class breakdown

In [19]:
for run in RUNS:
    ev = test_eval[run["name"]]
    print(f"--- {run['name']} " + "-" * (78 - len(run['name'])))
    print(classification_report(ev["targets"], ev["preds"], target_names=LABELS, digits=4))

per_class_f1 = pd.DataFrame(
    {run["name"]: f1_score(test_eval[run["name"]]["targets"], test_eval[run["name"]]["preds"],
                           average=None, labels=range(N_CLASSES)) for run in RUNS},
    index=LABELS)
print("Per-class F1\n")
print(per_class_f1.round(4).to_string())
print(f"\nHardest class for both models: {per_class_f1.mean(axis=1).idxmin()} "
      f"(mean F1 {per_class_f1.mean(axis=1).min():.4f})")

--- BiLSTM ------------------------------------------------------------------------
              precision    recall  f1-score   support

       World     0.9366    0.8789    0.9069      1900
      Sports     0.9459    0.9853    0.9652      1900
    Business     0.8850    0.8705    0.8777      1900
    Sci/Tech     0.8791    0.9111    0.8948      1900

    accuracy                         0.9114      7600
   macro avg     0.9117    0.9114    0.9111      7600
weighted avg     0.9117    0.9114    0.9111      7600

--- Transformer -------------------------------------------------------------------
              precision    recall  f1-score   support

       World     0.9227    0.9053    0.9139      1900
      Sports     0.9553    0.9674    0.9613      1900
    Business     0.8889    0.8674    0.8780      1900
    Sci/Tech     0.8754    0.9021    0.8885      1900

    accuracy                         0.9105      7600
   macro avg     0.9106    0.9105    0.9104      7600
weighted avg     

In [20]:
# Grouped bars: colour carries the model (fixed slots), direct value labels on every bar
# so identity and magnitude are never colour-alone.
fig, ax = plt.subplots(figsize=(7.6, 3.8))
y = np.arange(N_CLASSES)
bar_h = 0.36
gap_px = 0.02                                  # 2px-equivalent surface gap between adjacent bars

for k, run in enumerate(RUNS):
    name = run["name"]
    offs = (k - 0.5) * (bar_h + gap_px)
    bars = ax.barh(y + offs, per_class_f1[name].values, height=bar_h,
                   color=SERIES[name], label=name, zorder=2)
    for rect, v in zip(bars, per_class_f1[name].values):
        ax.text(v - 0.012, rect.get_y() + rect.get_height() / 2, f"{v:.3f}",
                ha="right", va="center", fontsize=8.5, color="#ffffff", fontweight="bold")

ax.set(title="Per-class F1 on the held-out test set", xlabel="F1", xlim=(0, 1.0))
ax.set_yticks(y, LABELS)
ax.invert_yaxis()
# Legend sits on the title row, outside the plot area: the bars reach ~0.96 so
# there is no in-axes space for it that does not cover data.
ax.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=2)
recede(ax, axis="x")
fig.tight_layout(); save_fig(fig, "fig1_per_class_f1.png"); plt.show()

saved -> artifacts/fig1_per_class_f1.png


### 7.2 Confusion matrices

Row-normalised (each row sums to 100%), so the colour reads as "of the true class, what fraction
went where". Both panels share one sequential blue ramp and one 0–100% scale, so they are directly
comparable.

In [21]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.5))
for ax, run in zip(axes, RUNS):
    ev = test_eval[run["name"]]
    cm = confusion_matrix(ev["targets"], ev["preds"], labels=range(N_CLASSES))
    cmn = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cmn, cmap=BLUE_SEQ, vmin=0.0, vmax=1.0)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{cmn[i, j]:.1%}\n{cm[i, j]}", ha="center", va="center", fontsize=8.5,
                    color="#ffffff" if cmn[i, j] > 0.55 else INK["primary"])
    ax.set(title=f"{run['name']} — {ev['accuracy']:.2%} accuracy",
           xlabel="predicted", ylabel="true" if ax is axes[0] else "")
    ax.set_xticks(range(N_CLASSES), LABELS, rotation=20, ha="right")
    ax.set_yticks(range(N_CLASSES), LABELS if ax is axes[0] else [""] * N_CLASSES)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("share of true class", color=INK["secondary"], fontsize=9)
cbar.outline.set_visible(False)
save_fig(fig, "fig2_confusion_matrices.png"); plt.show()

# Quantify the directional confusion that dominates AG News.
for run in RUNS:
    ev = test_eval[run["name"]]
    cm = confusion_matrix(ev["targets"], ev["preds"], labels=range(N_CLASSES))
    cmn = cm / cm.sum(axis=1, keepdims=True)
    off = [(cmn[i, j], LABELS[i], LABELS[j]) for i in range(N_CLASSES)
           for j in range(N_CLASSES) if i != j]
    v, a, b = max(off)
    print(f"{run['name']:<12} largest confusion: {a} -> {b} at {v:.1%}")

saved -> artifacts/fig2_confusion_matrices.png
BiLSTM       largest confusion: Business -> Sci/Tech at 9.2%
Transformer  largest confusion: Business -> Sci/Tech at 9.6%


### 7.3 Learning curves

In [22]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.8))
for run in RUNS:
    h, name = run["history"], run["name"]
    ep = np.arange(1, len(h["val_loss"]) + 1)
    axes[0].plot(ep, h["val_loss"], marker="o", color=SERIES[name], label=name, zorder=3)
    axes[0].plot(ep, h["train_loss"], marker="", linestyle=":", linewidth=1.6,
                 color=SERIES[name], alpha=0.75, zorder=2)
    axes[1].plot(ep, h["val_macro_f1"], marker="o", color=SERIES[name], label=name, zorder=3)
    # Direct-label the endpoint only, rather than annotating every point.
    axes[1].annotate(f"{h['val_macro_f1'][-1]:.3f}", xy=(ep[-1], h["val_macro_f1"][-1]),
                     xytext=(4, 0), textcoords="offset points", va="center",
                     fontsize=9, color=INK["secondary"])

axes[0].set(title="Loss per epoch  (solid = validation, dotted = train)",
            xlabel="epoch", ylabel="cross-entropy")
axes[1].set(title="Validation macro-F1 per epoch", xlabel="epoch", ylabel="macro-F1")
for ax in axes:
    ax.legend(loc="best")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    recede(ax)
fig.tight_layout(); save_fig(fig, "fig3_learning_curves.png"); plt.show()

saved -> artifacts/fig3_learning_curves.png


### 7.4 Computational cost

The parameter counts are reported both with and without the embedding table. The table is shared
in size between the two models and dominates both totals, so the **non-embedding** column is the
honest measure of architectural capacity.

In [23]:
@torch.no_grad()
def inference_latency(model, dataloader, repeats: int = 2) -> float:
    # ms per 1,000 documents, averaged over full passes of the test set.
    model.eval()
    n = sum(len(b[2]) for b in dataloader)
    best = math.inf
    for _ in range(repeats):
        t0 = time.time()
        for ids, lens, _ in dataloader:
            model(ids.to(DEVICE), lens.to(DEVICE))
        best = min(best, time.time() - t0)
    return best / n * 1000 * 1000


cost = pd.DataFrame([{
    "Model": run["name"],
    "Params (total)": run["params"]["total"],
    "Params (excl. embedding)": run["params"]["non_embedding"],
    "Epochs run": len(run["history"]["epoch_seconds"]),
    "Sec / epoch": run["seconds_per_epoch"],
    "Total train sec": run["total_seconds"],
    "Inference ms / 1k docs": inference_latency(run["model"], test_dl),
    "Test accuracy": test_eval[run["name"]]["accuracy"],
} for run in RUNS]).set_index("Model")

print("Computational cost\n")
print(cost.round(2).to_string())

sp = cost["Sec / epoch"]
FASTER  = sp.idxmin()
SLOWER  = sp.idxmax()
RATIO   = sp.max() / sp.min()
print(f"\nThe {SLOWER} takes {RATIO:.2f}x longer per epoch than the {FASTER} "
      f"({sp[SLOWER]:.1f}s vs {sp[FASTER]:.1f}s).")
print("Both models run the same data through the same harness on the same CPU, so the difference is")
print("structural: the Transformer evaluates all positions in a few batched matmuls, whereas the")
print(f"BiLSTM must execute {MAX_LEN} strictly sequential cell steps that cannot be parallelised.")
print("Note the constant factors: at this sequence length the Transformer does more arithmetic in")
print("total, so which model wins the wall clock depends on how much the sequential Python-level")
print("recurrence costs relative to that. The recurrent model's disadvantage grows with MAX_LEN.")

Computational cost

             Params (total)  Params (excl. embedding)  Epochs run  Sec / epoch  Total train sec  Inference ms / 1k docs  Test accuracy
Model                                                                                                                                 
BiLSTM              4612612                    265220           3        41.19           126.05                  137.57           0.91
Transformer         4613124                    265732           5        68.63           353.33                  334.70           0.91

The Transformer takes 1.67x longer per epoch than the BiLSTM (68.6s vs 41.2s).
Both models run the same data through the same harness on the same CPU, so the difference is
structural: the Transformer evaluates all positions in a few batched matmuls, whereas the
BiLSTM must execute 64 strictly sequential cell steps that cannot be parallelised.
Note the constant factors: at this sequence length the Transformer does more arithmetic in
tot

### 7.5 Where the models disagree

Accuracy alone cannot tell whether two models fail in the same way. McNemar's test looks only at
the **discordant** test documents — the ones exactly one model gets right — and asks whether that
split is distinguishable from a coin flip. Without it, a sub-point accuracy gap is easy to
over-read.

In [24]:
pred_a, pred_b = test_eval["BiLSTM"]["preds"], test_eval["Transformer"]["preds"]
truth = test_eval["BiLSTM"]["targets"]
assert np.array_equal(truth, test_eval["Transformer"]["targets"]), "evaluation order diverged"

ok_a, ok_b = pred_a == truth, pred_b == truth
both_right, both_wrong = int((ok_a & ok_b).sum()), int((~ok_a & ~ok_b).sum())
only_a, only_b = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())

print(pd.DataFrame([[both_right, only_a], [only_b, both_wrong]],
                   index=["Transformer correct", "Transformer wrong"],
                   columns=["BiLSTM correct", "BiLSTM wrong"]).to_string())

test = binomtest(only_b, only_a + only_b, p=0.5)
print(f"\nDiscordant documents: {only_a + only_b:,} of {len(truth):,} "
      f"({only_a:,} only-BiLSTM, {only_b:,} only-Transformer)")
print(f"McNemar (exact binomial), p = {test.pvalue:.4g}")
SIGNIF = test.pvalue < 0.05
print(f"At alpha=0.05 the difference IS{'' if SIGNIF else ' NOT'} statistically significant.")
print(f"Agreement rate between the two models: {(pred_a == pred_b).mean():.2%}")

                     BiLSTM correct  BiLSTM wrong
Transformer correct            6740           187
Transformer wrong               180           493

Discordant documents: 367 of 7,600 (187 only-BiLSTM, 180 only-Transformer)
McNemar (exact binomial), p = 0.7542
At alpha=0.05 the difference IS NOT statistically significant.
Agreement rate between the two models: 94.78%


In [25]:
# Qualitative error inspection on the raw test text.
test_texts = test_df.text.tolist()
rng = np.random.default_rng(SEED)


def show(indices, heading):
    print(f"\n{heading}\n" + "-" * 100)
    for i in indices:
        print(f"  true={LABELS[truth[i]]:<9} BiLSTM={LABELS[pred_a[i]]:<9} "
              f"Transformer={LABELS[pred_b[i]]:<9}")
        print(f"    {test_texts[i][:118]}")


both_wrong_idx = np.where(~ok_a & ~ok_b)[0]
disagree_idx   = np.where(pred_a != pred_b)[0]
show(rng.choice(both_wrong_idx, size=min(8, len(both_wrong_idx)), replace=False),
     "Documents BOTH models get wrong (genuinely ambiguous topic boundaries)")
show(rng.choice(disagree_idx, size=min(8, len(disagree_idx)), replace=False),
     "Documents where the two models DISAGREE")


Documents BOTH models get wrong (genuinely ambiguous topic boundaries)
----------------------------------------------------------------------------------------------------
  true=Sci/Tech  BiLSTM=World     Transformer=World    
    U.N. Urges Funds to Curb African Locusts (AP). AP - With swarms of locusts threatening crops in a number of African co
  true=World     BiLSTM=Sports    Transformer=Sports   
    Argentina Basketball Coach Magnano Quits. Ruben Magnano, who coached Argentina to the Olympic basketball gold medal in
  true=Sci/Tech  BiLSTM=World     Transformer=World    
    Echoes Repeats Success. Don't junk that GameCube: Metroid Prime 2 provides gorgeous atmosphere, a sweet score and fun 
  true=Sci/Tech  BiLSTM=World     Transformer=World    
    Singapore Telecom ties up with Malaysia's TIME dotcom (AFP). AFP - Singapore Telecommunications (SingTel) said it has 
  true=Business  BiLSTM=Sports    Transformer=Sports   
    Brewers buyer expected to step out of the shadows M

In [26]:
# Persist everything needed to reproduce the tables and figures without retraining.
summary = {
    "config": {k: globals()[k] for k in
               ["SMOKE_TEST", "SEED", "SUBSET_SIZE", "VAL_FRACTION", "MAX_LEN", "MIN_FREQ",
                "BATCH_SIZE", "MAX_EPOCHS", "PATIENCE", "LR_BILSTM", "LR_TRANSFORMER",
                "EMB_DIM", "LSTM_HIDDEN", "D_MODEL", "N_HEADS", "N_LAYERS", "D_FF"]},
    "vocab_size": VOCAB_SIZE,
    "splits": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "correctness_checks": {k: bool(v[0]) for k, v in results.items()},
    "models": {run["name"]: {
        "lr": run["lr"], "params": run["params"], "history": run["history"],
        "best_epoch": run["best_epoch"], "best_val_macro_f1": run["best_val_macro_f1"],
        "total_seconds": run["total_seconds"], "seconds_per_epoch": run["seconds_per_epoch"],
        "test": {k: float(test_eval[run["name"]][k])
                 for k in ["accuracy", "macro_f1", "weighted_f1", "loss"]},
        "per_class_f1": per_class_f1[run["name"]].round(6).to_dict(),
    } for run in RUNS},
    "mcnemar": {"only_bilstm_correct": only_a, "only_transformer_correct": only_b,
                "both_correct": both_right, "both_wrong": both_wrong,
                "p_value": float(test.pvalue), "significant_at_0.05": bool(SIGNIF)},
}
with open(Path(ART_DIR) / "metrics.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print(f"Wrote {Path(ART_DIR) / 'metrics.json'}")
print(f"Artifacts: {sorted(p.name for p in Path(ART_DIR).iterdir())}")

Wrote artifacts/metrics.json
Artifacts: ['bilstm_best.pt', 'fig0_token_lengths.png', 'fig1_per_class_f1.png', 'fig2_confusion_matrices.png', 'fig3_learning_curves.png', 'metrics.json', 'transformer_best.pt']


## 8. Conclusion

### Answering the research question

> *How do recurrent networks and Transformers compare at classifying short news texts?*

**They land in the same accuracy band, and they get there by different computational routes.**
The cell below prints the exact figures from this run; the interpretation below is written to hold
whichever way the measured numbers fall.

**1. Accuracy is close, and the gap needs a significance test to interpret.** Both architectures
reach the high-eighties / low-nineties on AG News, within roughly a point of each other. McNemar's
test in 7.5 is what decides whether that margin means anything: the two models agree on the large
majority of test documents, and the discordant set is where the real difference lives.

**2. The reason the gap is small is the "short" in short news texts.** AG News documents are a
headline plus a one-sentence blurb — a median of ~37 tokens. Self-attention's structural advantage
is modelling dependencies between distant positions at constant path length, and in a 40-token
document there are barely any distant positions to connect. Topic signal here is carried largely by
lexical evidence ("Reuters", "quarterly", "coach", "researchers") that a bidirectional recurrence
captures just as well. This is the key qualification on any claim that Transformers simply beat
RNNs at text classification: on this input length, the architectural advantage has little to bite on.

**3. The real difference is computational structure, and it is a question of scale.** The BiLSTM's
`MAX_LEN` cell steps are *strictly sequential* — every timestep waits on the previous one — while
the encoder evaluates all positions in a handful of batched matmuls. That is the standard argument
for Transformers, and it is an argument about *asymptotics*: the recurrent cost grows linearly in
sequence length and cannot be parallelised away, whereas attention trades that for more total
arithmetic that hardware can parallelise.

The measured per-epoch figures in 7.4 are therefore worth reading carefully rather than as a
verdict. At `MAX_LEN = 64` on CPU the two effects are of comparable size — the Transformer does
more FLOPs per document, the BiLSTM pays 64 sequential dispatches — so the constant factors, not
the asymptotics, decide this particular wall clock. The architectural advantage that makes
Transformers dominant is real but mostly *outside* this regime: it shows up at long sequence
lengths and on parallel hardware, neither of which a 40-token headline on 6 CPU threads provides.

**4. Both models fail on the same boundary.** The confusion matrices put most residual error in the
Business ↔ Sci/Tech pair. That reflects the labelling: a story about a semiconductor firm's earnings
is legitimately both. Error of this kind is a property of the task, not of either architecture, and
it caps what any model can achieve here.

### Limitations

- A **single seed** per architecture. The accuracy gap is of the same order as run-to-run seed
  variance, so a stronger claim needs several seeds and a confidence interval.
- **Controlled subset** of 60,000 of the 120,000 training records, per the brief; both models would
  improve somewhat with the full set, and the more parameter-hungry Transformer likely more so.
- **Randomly initialised embeddings** and a small 2-layer encoder. This measures the architectures
  at matched capacity trained from scratch — it says nothing about pretrained Transformers, where
  transferred representations, not the architecture, are the dominant factor.
- `MAX_LEN = 64` truncates a small tail of documents, slightly favouring neither model.

In [27]:
acc_a = test_eval["BiLSTM"]["accuracy"]
acc_b = test_eval["Transformer"]["accuracy"]
f1_a, f1_b = test_eval["BiLSTM"]["macro_f1"], test_eval["Transformer"]["macro_f1"]
win = "Transformer" if acc_b > acc_a else "BiLSTM"
hardest = per_class_f1.mean(axis=1).idxmin()
acc_close = abs(acc_b - acc_a) < 0.01          # within 1 percentage point

print("=" * 84)
print("SUMMARY — AG News topic classification, from-scratch BiLSTM vs. Transformer encoder")
print("=" * 84)
print(f"  Controlled subset  : {len(train_df):,} train / {len(val_df):,} val, "
      f"vocab {VOCAB_SIZE:,}, max_len {MAX_LEN}")
print(f"  Held-out test set  : {len(test_df):,} official AG News test records")
print()
print(f"  BiLSTM             : accuracy {acc_a:.4f} | macro-F1 {f1_a:.4f} | "
      f"{run_bilstm['seconds_per_epoch']:.1f}s/epoch | "
      f"{run_bilstm['params']['non_embedding']:,} non-embedding params")
print(f"  Transformer        : accuracy {acc_b:.4f} | macro-F1 {f1_b:.4f} | "
      f"{run_transformer['seconds_per_epoch']:.1f}s/epoch | "
      f"{run_transformer['params']['non_embedding']:,} non-embedding params")
print()
print(f"  Accuracy gap       : {abs(acc_b - acc_a) * 100:.2f} pp in favour of the {win}")
print(f"  McNemar p-value    : {test.pvalue:.4g} -> "
      f"{'significant' if SIGNIF else 'NOT significant'} at alpha=0.05")
print(f"  Model agreement    : {(pred_a == pred_b).mean():.2%} of test documents")
print(f"  Speed              : the {SLOWER} is {RATIO:.2f}x slower per epoch "
      f"({sp[SLOWER]:.1f}s vs {sp[FASTER]:.1f}s)")
print(f"  Hardest class      : {hardest} (Business <-> Sci/Tech boundary)")
print("=" * 84)
acc_phrase = ("near-identical accuracy" if acc_close else
              f"a {abs(acc_b - acc_a) * 100:.2f} pp accuracy edge to the {win}")
print(f"  VERDICT: {acc_phrase}; the {FASTER} trains {RATIO:.2f}x faster per epoch here.")
print(f"           Documents this short (median {int(np.median(lengths))} tokens) give self-attention")
print("           little long-range structure to exploit, so the recurrent model stays")
print("           competitive on quality. See section 8 on why the cost comparison is")
print("           a statement about this scale, not about the architectures in general.")
print("=" * 84)

SUMMARY — AG News topic classification, from-scratch BiLSTM vs. Transformer encoder
  Controlled subset  : 54,000 train / 6,000 val, vocab 33,964, max_len 64
  Held-out test set  : 7,600 official AG News test records

  BiLSTM             : accuracy 0.9114 | macro-F1 0.9111 | 41.2s/epoch | 265,220 non-embedding params
  Transformer        : accuracy 0.9105 | macro-F1 0.9104 | 68.6s/epoch | 265,732 non-embedding params

  Accuracy gap       : 0.09 pp in favour of the BiLSTM
  McNemar p-value    : 0.7542 -> NOT significant at alpha=0.05
  Model agreement    : 94.78% of test documents
  Speed              : the Transformer is 1.67x slower per epoch (68.6s vs 41.2s)
  Hardest class      : Business (Business <-> Sci/Tech boundary)
  VERDICT: near-identical accuracy; the BiLSTM trains 1.67x faster per epoch here.
           Documents this short (median 39 tokens) give self-attention
           little long-range structure to exploit, so the recurrent model stays
           competitive on qual